In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
#1. Preparing historical option observations with inputs such as moneyness, maturity, implied volatility, Black-Scholes delta, realized volatility, option price, volume, and recent stock returns.
options = pd.read_csv("../../data/backtest_options.csv")
stock = pd.read_csv("../../data/spy_stock_prices.csv")

options["date"] = pd.to_datetime(options["date"])
options["expiration"] = pd.to_datetime(options["expiration"])
stock["date"] = pd.to_datetime(stock["date"])

# Option features
options["mid_price"] = (options["bid"] + options["ask"]) / 2
options["T"] = (options["expiration"] - options["date"]).dt.days / 365
options["moneyness"] = options["strike"] / options["underlying_price"]
options["log_moneyness"] = np.log(options["moneyness"])

# Stock-volatility features
stock["log_return"] = np.log(stock["close"] / stock["close"].shift(1))
stock["realized_vol_20"] = stock["log_return"].rolling(20).std() * np.sqrt(252)
stock["realized_vol_60"] = stock["log_return"].rolling(60).std() * np.sqrt(252)

options = options.merge(
    stock[["date", "log_return", "realized_vol_20", "realized_vol_60"]],
    on="date",
    how="left",
)

print(options.head())

        date expiration option_type  strike        bid        ask  lastPrice  \
0 2020-02-19 2020-05-07        call     340  50.053250  50.556298  50.304774   
1 2020-02-20 2020-05-07        call     340  48.304017  48.789484  48.546751   
2 2020-02-21 2020-05-07        call     340  46.835767  47.306478  47.071122   
3 2020-02-24 2020-05-07        call     340  39.935645  40.337008  40.136326   
4 2020-02-25 2020-05-07        call     340  33.647606  33.985773  33.816689   

   volume  openInterest  impliedVolatility  ...  mid_price  days_to_expiry  \
0     500          5000                0.8  ...  50.304774              78   
1     500          5000                0.8  ...  48.546751              77   
2     500          5000                0.8  ...  47.071122              76   
3     500          5000                0.8  ...  40.136326              73   
4     500          5000                0.8  ...  33.816689              72   

          T  moneyness  log_moneyness  bid_ask_spr

In [2]:
#2. Creating the target hedge ratio from consecutive prices:
# Sort so each option contract is in time order
options = options.sort_values(["expiration", "strike", "date"])

# Next day's option price and stock price for the same contract
options["next_mid_price"] = options.groupby(["expiration", "strike"])["mid_price"].shift(-1)
options["next_stock_price"] = options.groupby(["expiration", "strike"])["underlying_price"].shift(-1)

# Realized hedge ratio:
# how much the option price changed relative to the stock price change
options["realized_delta"] = (
    (options["next_mid_price"] - options["mid_price"]) /
    (options["next_stock_price"] - options["underlying_price"])
)

# Clean bad/infinite values
options = options.replace([np.inf, -np.inf], np.nan)
options = options.dropna(subset=["realized_delta"])

# Optional: clip extreme noisy values
options["realized_delta"] = options["realized_delta"].clip(-2, 2)

print(options[["date", "strike", "mid_price", "underlying_price", "realized_delta"]].head())

        date  strike  mid_price  underlying_price  realized_delta
0 2020-02-19     340  50.304774            338.32        1.321822
1 2020-02-20     340  48.546751            336.99        0.416844
2 2020-02-21     340  47.071122            333.45        0.628721
3 2020-02-24     340  40.136326            322.42        0.642893
4 2020-02-25     340  33.816689            312.59        0.571423


In [3]:
#3. Splitting chronologically into training and out-of-sample test data, then standardizing the input features.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Features the neural network will use to predict realized_delta
feature_cols = [
    "moneyness",
    "log_moneyness",
    "T",
    "impliedVolatility",
    "realized_vol_20",
    "realized_vol_60",
    "log_return",
    "mid_price",
]

# Keep only rows with complete feature + target data
model_data = options.dropna(subset=feature_cols + ["realized_delta"]).copy()

# Sort by date so the split respects time order
model_data = model_data.sort_values("date")

X = model_data[feature_cols]
y = model_data["realized_delta"]

# Chronological split: first 80% train, last 20% test
split_idx = int(len(model_data) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Standardize features using only the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Features used:", feature_cols)

ValueError: Found array with 0 sample(s) (shape=(0, 8)) while a minimum of 1 is required by StandardScaler.

In [ ]:
#4. Training a neural network to map the market features directly to a predicted option delta.
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Build the neural network
nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42,
)

# Train the model
nn_model.fit(X_train_scaled, y_train)

# Predict realized delta on the test set
y_pred = nn_model.predict(X_test_scaled)

# Evaluate prediction accuracy
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("Test MSE:", mse)
print("Test MAE:", mae)
print("First 10 predicted deltas:")
print(y_pred[:10])

Test MSE: 1.1958747936132272
Test MAE: 0.9062108639635763
First 10 predicted deltas:
[1.35460978 1.22220585 1.06349398 0.86945012 0.91364042 2.12942315
 2.21176869 1.07626579 0.54934991 0.92207496]


In [ ]:
#5. Predicting delta each test day and buying approximately 100 \times \Delta_NN
# Predict neural-network deltas on the test data
test_data = model_data.iloc[split_idx:].copy()

test_data["nn_delta"] = nn_model.predict(X_test_scaled)

# For one short call option contract:
option_quantity = -1
contract_multiplier = 100

# Delta exposure from the short call position
test_data["option_position_delta"] = (
    option_quantity * test_data["nn_delta"] * contract_multiplier
)

# Number of SPY shares needed to offset the option delta
test_data["target_shares"] = -test_data["option_position_delta"]

print(
    test_data[
        ["date", "strike", "underlying_price", "nn_delta", "target_shares"]
    ].head()
)

          date  strike  underlying_price  nn_delta  target_shares
183 2026-04-02     510        551.786756  1.354610     135.460978
184 2026-04-03     510        551.375756  1.222206     122.220585
185 2026-04-06     510        549.512505  1.063494     106.349398
186 2026-04-07     510        544.729423  0.869450      86.945012
187 2026-04-08     510        538.184828  0.913640      91.364042


In [ ]:
#6. Rebalancing daily as the network’s predicted delta changes, then comparing its hedging error and costs with the Dupire hedge.
test_data = test_data.sort_values("date").copy()

current_shares = 0
rebalance_rows = []

for _, row in test_data.iterrows():
    date = row["date"]
    stock_price = row["underlying_price"]
    target_shares = row["target_shares"]

    # Shares to buy/sell today
    shares_to_trade = target_shares - current_shares

    # Dollar cost of the trade
    trade_value = shares_to_trade * stock_price

    rebalance_rows.append({
        "date": date,
        "stock_price": stock_price,
        "nn_delta": row["nn_delta"],
        "target_shares": target_shares,
        "current_shares_before_trade": current_shares,
        "shares_to_trade": shares_to_trade,
        "trade_value": trade_value,
    })

    # After rebalancing, today's target becomes the current position
    current_shares = target_shares

rebalance_df = pd.DataFrame(rebalance_rows)

print(rebalance_df.head())

        date  stock_price  nn_delta  target_shares  \
0 2026-04-02   551.786756  1.354610     135.460978   
1 2026-04-03   551.375756  1.222206     122.220585   
2 2026-04-06   549.512505  1.063494     106.349398   
3 2026-04-07   544.729423  0.869450      86.945012   
4 2026-04-08   538.184828  0.913640      91.364042   

   current_shares_before_trade  shares_to_trade   trade_value  
0                     0.000000       135.460978  74745.573578  
1                   135.460978       -13.240393  -7300.431596  
2                   122.220585       -15.871187  -8721.415871  
3                   106.349398       -19.404386 -10570.139921  
4                    86.945012         4.419030   2378.254797  
